# pawn covers
This notebook shows how to create a pawn cover so that queens can cross the board.

In [ ]:
#| default_exp game/mechanics/pawncover

In [ ]:
#| export
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *
import pandas as pd
import numpy as np

In [ ]:
#| export
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory
from HexMagic.geology import Geology, DrainageBasins, SoilSystem
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets

from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

In [ ]:
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay, food_triage, pre_march_distribute, compose_callbacks

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay

In [ ]:
import pyomo.environ as pyo
import optuna

## Food

Clean results:

    Tier-sum ≥ 7 rule holds perfectly — every sum-7 combo passes, sum-6 fails
    Tight margin at sum=7 — only 0.7 surplus (7.7 vs 7.0), so it works but there's no slack. Good tension.
    Solo T7 pawn feeds the queen — strategic chokepoint mechanic works
    Pawn survives on T1 — pawns never starve, even on the worst land
    10× diet ratio — queen is very expensive, which makes escort logistics the core game

The numbers are all round and easy for a player to reason about: "I need my two pawns' tiers to add up to 7." That's a great mental model.

One thing to note: we don't have knight/rook profiles in V2 yet. Worth adding those when you're ready, but for now this is solid for testing the PCR. Want to rerun the route with this profile and get the overlay working?

In [ ]:
#| export
FOOD_PROFILE_V2 = {
    'pawn_diet':       0.7,
    'pawn_harvest':    1.3,
    'pawn_capacity':  14.0,
    'pawn_sight':      4,
    'queen_diet':      7.0,
    'queen_harvest':   0.2,
    'queen_capacity': 24.0,
    'bishop_diet':     3.5,
    'bishop_harvest':  0.7,
    'bishop_capacity': 140.0,
}

def apply_food_profile(pieces, profile=None):
    """Apply food profile to a list of pieces."""
    p = profile or FOOD_PROFILE_V2
    for piece in pieces:
        t = piece.piece_type.name.lower()
        for attr, field in [('diet','diet'), ('harvest','harvest_strength'),
                            ('capacity','food_capacity'), ('sight','sight')]:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, field, p[key])
        # Fill to capacity
        piece.food = piece.food_capacity

In [ ]:
@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    pieces = []
    for country in self.board.kingdoms:
        for city in country.settlements:
            for piece in city.citizens:
                profile = PROFILES.get(piece.piece_type)
                if profile is None:
                    continue
                piece.diet, piece.harvest_strength, piece.food_capacity = profile
                piece.food = piece.food_capacity  # start full
                count[piece.piece_type.name] += 1
                pieces.append(piece)

    apply_food_profile(pieces,FOOD_PROFILE_V2)
    
    # Apply soil scaling if needed

    fy = FoodYield(self.terr, self.basin)
    if soil_scale != 1.0:
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()

    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
    return fy

In [ ]:
p = FOOD_PROFILE_V2
h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']

print("Tier-sum rule (2 pawns, need sum ≥ 7):")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (3,4), (7,0)]:
    s = ta + tb
    exp = h*s - 2*d if tb > 0 else h*ta - d
    label = f"T{ta}+T{tb}" if tb > 0 else f"Solo T{ta}"
    print(f"  {label:10s} sum={s:2d}  export={exp:.1f}  {'✅' if exp >= Q else '❌'}")
print(f"\n  Pawn on T1: {h*1:.1f} vs diet {d:.1f} → {'survives' if h >= d else 'starves'}")
print(f"  Queen/pawn ratio: {Q/d:.1f}×")

## coverage

In [ ]:
#| export
def _solver_ok(result):
    return (result.solver.status == pyo.SolverStatus.ok and
            result.solver.termination_condition == pyo.TerminationCondition.optimal)

In [ ]:
#| export
def _find_hungry(alive, grid, piece, safety_turns=2):
    """Find the most desperate piece, sorted by urgency."""
    hungry = []
    for p in alive:
        if p is piece: continue
        if p.health <= 0: continue
        turns_left = p.food / max(p.diet, 0.1)
        if turns_left < safety_turns * 3:  # worth helping
            hungry.append((p, turns_left))
    hungry.sort(key=lambda h: h[1])  # most desperate first
    return hungry


def _adjacent(grid, loc_a, loc_b):
    """True if same hex or neighbors."""
    if loc_a == loc_b: return True
    return loc_b in grid.neighborsOf(loc_a)


def _donor_surplus(piece, safety_turns=2):
    """How much food this piece can spare."""
    keep = piece.diet * safety_turns
    return max(0, piece.food - keep)


def food_redistribute(report: TurnReport):
    """Squad-level food sharing — three tiers of urgency.
    
    Tier 1: Direct transfer (adjacent, no instruction cost)
    Tier 2: Rotate toward hungry (prepend ROT, then GIVE next turn)
    Tier 3: Move toward hungry (prepend FORWARD steps toward them)
    
    Only donors with surplus above their own safety margin participate.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # ── Tier 1: Direct transfers (free, every turn) ──
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break
    
    # ── Tier 2 & 3: Only for pieces with surplus that aren't feeding ──
    # Check if anyone is still desperate after direct transfers
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # already handled in tier 1
        
        # ── Tier 2: Rotate to face the hungry piece ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            if diff <= 3:
                rot = Instruction.ROT_R.value
            else:
                rot = Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            # Only insert if not already rotating toward them
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue  # already rotating, don't stack
            
            donor.instructions.rules.insert(cursor, rot)
            continue  # rotate this turn, give next turn
        
        # ── Tier 3: Facing correct but not adjacent — move toward ──
        # Only move if the receiver is truly desperate (< 2 turns)
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        # Check if forward movement gets us closer
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        # Peek at hex in front
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue  # already moving
                
                donor.instructions.rules.insert(cursor, Instruction.FORWARD.value)


def food_harvest_check(report: TurnReport):
    """Prepend HARVEST when a piece is on a food tile and hungry.
    
    Capped at 1 insert per turn — won't corrupt instruction lists.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0: continue
        if piece.food >= piece.food_capacity * 0.8: continue
        
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier < 1: continue
        
        rules = piece.instructions.rules
        cursor = piece.instructions.cursor
        if cursor < len(rules) and rules[cursor] == Instruction.HARVEST.value:
            continue
        
        piece.instructions.rules.insert(cursor, Instruction.HARVEST.value)


def food_triage_v2(report: TurnReport):
    """Combined: harvest opportunities, then redistribute/rescue."""
    food_harvest_check(report)
    food_redistribute(report)

In [ ]:
#| export
def food_redistribute_adjacent(report: TurnReport):
    """Tier 1: Direct adjacent transfers. No instructions touched.
    
    Safe to call every turn, even during march.
    Moves food from anyone with surplus to the most desperate neighbor.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    
    # Most desperate first
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        # Donors: adjacent pieces with surplus above safety margin
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break


def food_redistribute_rescue(report: TurnReport):
    """Tiers 2-3: Rotate/move toward hungry. Edits instructions.
    
    Only call during CAMP or emergency. Will disrupt march orders.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # Anyone still desperate after tier-1?
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # tier 1 already handled
        
        # ── Tier 2: Rotate toward hungry ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            rot = Instruction.ROT_R.value if diff <= 3 else Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue
            
            rules.insert(cursor, rot)
            continue
        
        # ── Tier 3: Move toward hungry (only if truly dying) ──
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue
                rules.insert(cursor, Instruction.FORWARD.value)

## layout

In [ ]:
#| export
def optimal_camp_layout(squad, center, grid, elevations, food_tiers,
                        countries=None, max_ring=3, elevation_mult=0.005):
    """Network-flow camp placement MIP.
    
    The hex neighborhood is a directed graph:
      - Nodes: candidate hexes (occupied by pieces)
      - Arcs: adjacency edges (or sight-cone edges for longer range)
      - Source nodes: pawns on food hexes (production = harvest × tier)
      - Relay nodes: bishops (high throughput, buffer capacity)
      - Sink nodes: queen/king (high diet, low harvest)
    
    Flow conservation at each occupied hex:
      production + inflow ≥ consumption + outflow
    
    Placement decides WHO goes WHERE.
    Flow decides HOW MUCH food moves along each arc.
    Objective: maximize net surplus (total prod - total diet),
               tiebreak by keeping high-combat pieces off food duty.
    """
    alive = squad.alive
    N = len(alive)
    
    # ── Candidate hexes (ring 0..max_ring around center) ──
    cands = []
    seen = set()
    for ring in range(0, max_ring + 1):
        ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
        for hp in ring_hps:
            idx = grid.hexposition_to_index(hp, center)
            if idx < 0 or idx in grid.invalidRegion or idx in seen: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            seen.add(idx)
            cands.append(idx)
    
    H = len(cands)
    tier = [int(food_tiers[c]) if 0 <= c < len(food_tiers) else 0
            for c in cands]
    hex_to_local = {c: i for i, c in enumerate(cands)}
    
    # ── Directed arcs: adjacency-based ──
    # (For sight-cone arcs, expand this with FOV precomputation)
    arcs = []
    for i in range(H):
        for nb in grid.neighborsOf(cands[i]):
            if nb in hex_to_local:
                j = hex_to_local[nb]
                arcs.append((i, j))
    
    # Piece parameters
    diet_p = [p.diet for p in alive]
    harv_p = [p.harvest_strength for p in alive]
    combat_p = [getattr(p, 'attack_strength', 0) for p in alive]
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.H = pyo.RangeSet(0, H - 1)
    m.A = pyo.Set(initialize=arcs)
    
    # ── Decision variables ──
    m.place = pyo.Var(m.P, m.H, domain=pyo.Binary)    # piece p on hex h
    m.flow  = pyo.Var(m.A, domain=pyo.NonNegativeReals) # food along arc
    
    # ── Placement constraints ──
    m.one_hex = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.place[p, h] for h in m.H) == 1)
    
    m.one_piece = pyo.Constraint(m.H, rule=lambda m, h:
        sum(m.place[p, h] for p in m.P) <= 1)
    
    # ── Production / consumption at each hex ──
    def _prod(m, h):
        """Harvest output: Σ_p place[p,h] × harv_p × tier[h]"""
        return sum(m.place[p, h] * harv_p[p] * tier[h] for p in m.P)
    
    def _cons(m, h):
        """Diet demand: Σ_p place[p,h] × diet_p"""
        return sum(m.place[p, h] * diet_p[p] for p in m.P)
    
    # ── Flow conservation at every hex ──
    # production + inflow ≥ consumption + outflow
    def balance(m, h):
        inflow  = sum(m.flow[j, h] for (j, hh) in arcs if hh == h)
        outflow = sum(m.flow[h, j] for (hh, j) in arcs if hh == h)
        return _prod(m, h) + inflow >= _cons(m, h) + outflow
    m.balance = pyo.Constraint(m.H, rule=balance)
    
    # ── Arc capacity: can only send from an occupied hex ──
    M_big = max(harv_p) * max(tier + [1]) * 2
    m.arc_send = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, i] for p in m.P))
    m.arc_recv = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, j] for p in m.P))
    
    # ── Objective ──
    # Primary: maximize total surplus (production - consumption)
    # Secondary: minimize combat value on food hexes (keep fighters free)
    α = 100  # surplus is much more important than combat tiebreak
    
    # A piece is "on food duty" if on a hex with tier > 0
    food_duty_combat = sum(
        combat_p[p] * m.place[p, h] * (1 if tier[h] > 0 else 0)
        for p in range(N) for h in range(H))
    
    m.obj = pyo.Objective(
        expr=α * (sum(_prod(m, h) for h in m.H) 
                - sum(_cons(m, h) for h in m.H))
             - food_duty_combat,
        sense=pyo.maximize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    if not _solver_ok(result):
        return None
    
    # ── Extract results ──
    placements = {}  # piece → hex_idx
    for p in m.P:
        for h in m.H:
            if pyo.value(m.place[p, h]) > 0.5:
                placements[alive[p].id] = cands[h]
                break
    
    flows = {}  # (hex_i, hex_j) → flow amount
    for (i, j) in arcs:
        v = pyo.value(m.flow[i, j])
        if v > 0.01:
            flows[(cands[i], cands[j])] = round(v, 2)
    
    total_prod = sum(pyo.value(_prod(m, h)) for h in m.H)
    total_cons = sum(pyo.value(_cons(m, h)) for h in m.H)
    
    return {
        'placements': placements,
        'flows': flows,
        'production': total_prod,
        'consumption': total_cons,
        'surplus': total_prod - total_cons,
        'cands': cands,
        'tiers': tier,
    }

In [ ]:
def smart_placement(settlement, squad, terrain, food_tiers, grid,
                       max_ring=10, camp_ring=3, top_k=5):
    """Place entire squad using optimal_camp_layout network-flow MIP.
    
    Phase 1: Score candidate centers cheaply (sum of nearby food tiers)
    Phase 2: Run full MIP on top-k candidates
    Phase 3: Apply best layout
    
    Returns the best result dict from optimal_camp_layout, or None.
    """
    center = settlement.location
    elevs = terrain.elevations
    
    # ── Phase 1: Cheap scoring of candidate centers ──
    candidates = []
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        
        # Quick heuristic: sum food tiers in the camp neighborhood
        score = 0
        land_count = 0
        for r in range(0, camp_ring + 1):
            ring_hps = [HexPosition.origin()] if r == 0 else HexPosition.origin().ring(r)
            for nhp in ring_hps:
                nidx = grid.hexposition_to_index(nhp, origin_index=idx)
                if nidx < 0 or elevs[nidx] <= 0: continue
                land_count += 1
                score += int(food_tiers[nidx]) if 0 <= nidx < len(food_tiers) else 0
        
        # Need enough land hexes for the squad
        if land_count >= len(squad.alive):
            candidates.append((idx, score))
    
    if not candidates:
        return None
    
    # Top-k by cheap score
    candidates.sort(key=lambda x: -x[1])
    shortlist = candidates[:top_k]
    
    # ── Phase 2: Full MIP on shortlist ──
    best_result = None
    best_surplus = -float('inf')
    best_center = center
    
    for cand_idx, cheap_score in shortlist:
        result = optimal_camp_layout(
            squad, cand_idx, grid, elevs, food_tiers,
            max_ring=camp_ring)
        
        if result is None: continue
        if result['surplus'] > best_surplus:
            best_surplus = result['surplus']
            best_result = result
            best_center = cand_idx
    
    if best_result is None:
        return None
    
    # ── Phase 3: Apply placements + facings ──
    for piece in squad.alive:
        hex_idx = best_result['placements'].get(piece.id)
        if hex_idx is not None:
            piece.location = hex_idx
            # Face toward camp center (for GIVE arcs)
            if hex_idx != best_center:
                piece.facing = _facing_toward(grid, hex_idx, best_center)
    
    best_result['center'] = best_center
    print(f"Smart placement v2: center={best_center}, "
          f"surplus={best_surplus:.1f} "
          f"(prod={best_result['production']:.1f}, "
          f"cons={best_result['consumption']:.1f}), "
          f"checked {len(shortlist)} candidates")
    
    return best_result

In [ ]:
#| export
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")

In [ ]:
### Show demo

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts()

In [ ]:
def rebootWorld():

    world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

    sampleMap = world.terrain
    sampleMap.repalette()

    for i in range(len(sampleMap.hexGrid.hexes)):
        sampleMap.hexGrid.hexes[i].label = str(i)

    myStuff.loadGeology(world)
    myStuff.grid.adjustRadius(40)
    for i in range(len(myStuff.grid.hexes)):
        myStuff.grid.hexes[i].label = str(i)

    return myStuff

In [ ]:
rebootWorld()

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    #SustainOverlay(queen=myStuff.queen),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

In [ ]:
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage_v2},
)

sim.run(num_turns=40)
sim.summary()

In [ ]:
#| export
def pawn_covered_route(start_hex, end_hex, grid, elevations, food_tiers,
                       pawn_harvest=2.0, pawn_diet=2.0, queen_diet=7.0,
                       cone_range=2, countries=None):
    """Find a route and place pawns to feed a queen traversing it.
    
    Returns dict with route, pawn placements, coverage info, or None.
    """
    from collections import deque
    
    parent = {start_hex: None}
    queue = deque([start_hex])
    while queue:
        cur = queue.popleft()
        if cur == end_hex: break
        for nb in grid.neighborsOf(cur):
            if nb in parent: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            parent[nb] = cur
            queue.append(nb)
    
    if end_hex not in parent:
        print(f"No land route ⬡{start_hex} → ⬡{end_hex}"); return None
    
    # Reconstruct path
    route = []
    cur = end_hex
    while cur is not None:
        route.append(cur)
        cur = parent[cur]
    route.reverse()
    route_set = set(route)
    W = len(route)
    
    # ── Candidate pawn positions: adjacent to route, not ON route, fertile ──
    candidates = []
    seen = set()
    for whex in route:
        for nb in grid.neighborsOf(whex):
            if nb in seen or nb in route_set: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            seen.add(nb)
            
            tier = int(food_tiers[nb]) if 0 <= nb < len(food_tiers) else 0
            if tier < 1: continue
            
            # Coverage: which route indices can this pawn reach?
            hp_nb = grid.index_to_hexposition(nb)
            coverage = set()
            for wi, whex2 in enumerate(route):
                hp_w = grid.index_to_hexposition(whex2)
                if hp_nb.distance(hp_w) <= cone_range:
                    coverage.add(wi)
            if not coverage: continue
            
            net_export = pawn_harvest * tier - pawn_diet  # surplus per turn
            candidates.append({
                'hex': nb, 'tier': tier, 'coverage': coverage,
                'export': max(0, net_export),
            })
    
    if not candidates:
        print("No fertile candidates adjacent to route"); return None
    
    # ── Set-cover MIP: minimum pawns to cover every route hex ──
    S = len(candidates)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)
    m.placed = pyo.Var(m.S, domain=pyo.Binary)
    
    m.cover = pyo.ConstraintList()
    uncoverable = set()
    for w in range(W):
        covers = [s for s in range(S)
                  if w in candidates[s]['coverage'] and candidates[s]['export'] > 0]
        if not covers:
            uncoverable.add(w); continue
        m.cover.add(
            sum(candidates[s]['export'] * m.placed[s] for s in covers) >= queen_diet)
    
    m.obj = pyo.Objective(
        expr=sum(m.placed[s] for s in m.S), sense=pyo.minimize)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable hexes"); return None
    
    # ── Extract placements ──
    placements = []
    for s in range(S):
        if pyo.value(m.placed[s]) > 0.5:
            placements.append(candidates[s])
    
    # ── Coverage check ──
    covered = set()
    for p in placements:
        covered |= p['coverage']
    
    gaps = [w for w in range(W) if w not in covered]
    
    print(f"PCR: route {W} hexes, {len(placements)} pawns, "
          f"{len(covered)}/{W} covered"
          + (f", {len(gaps)} gaps" if gaps else " ✅"))
    for p in placements:
        lo, hi = min(p['coverage']), max(p['coverage'])
        print(f"  ⬡{p['hex']} T{p['tier']} export={p['export']:.1f}/t  path[{lo}..{hi}]")
    
    return {
        'route': route,
        'placements': placements,
        'covered': covered,
        'gaps': gaps,
        'uncoverable': uncoverable,
        'queen_diet': queen_diet,
    }

In [ ]:
cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)
print(len(cities))

In [ ]:
start = cities[0].location
finish = cities[1].location
start = cities[0].location
finish = cities[1].location
print(f"Route: ⬡{start} → ⬡{finish}")

pcr = pawn_covered_route(
    start, finish,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

In [ ]:
rebootWorld()

In [ ]:
# ── Step 1: Create pawn pieces at PCR placements ──
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

# Fresh squad for the PCR
pcr_squad = Squad.squads(1, myStuff.country.flag)[0]

# Add the queen
queen = myStuff.queen
queen.location = pcr['route'][0]  # start of route
queen.food = FOOD_PROFILE_V2['queen_capacity']
pcr_squad.pieces.append(queen)

# Create pawns at each placement hex
for i, pl in enumerate(pcr['placements']):
    pawn = myStuff.home.recruit(PieceType.PAWN, flag=pcr_squad.flag)
    pawn.location = pl['hex']
    
    # Face toward nearest route hex
    hp_p = myStuff.grid.index_to_hexposition(pl['hex'])
    mid_w = sorted(pl['coverage'])[len(pl['coverage']) // 2]
    route_hex = pcr['route'][mid_w]
    pawn.facing = _facing_toward(myStuff.grid, pl['hex'], route_hex)
    
    # Instructions: harvest then give, on patrol (loop forever)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    
    pcr_squad.pieces.append(pawn)

# Apply V2 profile to all pieces in the squad
apply_food_profile(pcr_squad.pieces)

print(f"Squad: {len(pcr_squad.pieces)} pieces "
      f"({len(pcr_squad.alive)} alive)")
for p in pcr_squad.pieces:
    print(f"  {p.name:10s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")

In [ ]:
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    
    for i in range(len(route) - 1):
        # Which of the 6 directions goes from route[i] to route[i+1]?
        target_facing = None
        for d in range(6):
            nb_idx = grid.hexposition_to_index(dirs[d], route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate shortest direction
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")

In [ ]:
#| export
# ── Step 3: Run the simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=pcr_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[pcr_squad],
    callbacks={pcr_squad.id: food_redistribute_adjacent},
)

sim.run(num_turns=30)
sim.summary()

In [ ]:
#| export
show(sim)

In [ ]:
#| export
# ── Step 4: Check the queen's journey ──
queen_id = queen.id

rows = []
for snap in sim.snapshots:
    qs = snap['piece_states'].get(queen_id)
    if qs:
        rows.append({
            'turn': snap['turn'],
            'hex': qs['location'],
            'food': round(qs['food'], 2),
            'hp': round(qs['health'], 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"\nQueen arrived: {'✅' if queen.location == pcr['route'][-1] else '❌'}")
print(f"Final food: {queen.food:.1f}, HP: {queen.health}/{queen.max_health}")

# How many gives did the queen receive?
gives = sim.df[(sim.df['event'] == 'gave') & (sim.df['dst_piece_id'] == queen_id)]
print(f"Total gives received: {len(gives)}, total food: {gives['amount'].sum():.1f}")

In [ ]:
#| export
# What did the queen actually do each turn?
q_events = sim.df[sim.df['piece_id'] == queen_id].copy()
q_events = q_events[q_events['event'].isin(['moved', 'rotated', 'ate'])]
for turn, grp in q_events.groupby('turn'):
    if turn > 8: break
    moves = grp[grp['event'] == 'moved']
    rots = grp[grp['event'] == 'rotated']
    hexes = moves['dst_hex'].tolist()
    print(f"Turn {turn}: {len(rots)} rots, {len(moves)} moves → {hexes}  "
          f"(facing={queen.facing if turn == max(q_events.turn) else '?'})")

# Also check: is instruction cursor where we expect?
print(f"\nInstruction cursor: {queen.instructions.cursor}/{len(queen.instructions.rules)}")
print(f"Queen at: ⬡{queen.location}, facing: {queen.facing}")
print(f"Pawn at ⬡368: {any(p.location == 368 for p in pcr_squad.pieces if p != queen)}")

Does the pawn cover need to use correct food profile? we seem to have too many pawns which now have vision 4.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good catch — the PCR **is** already using `FOOD_PROFILE_V2` values (harvest=1.3, diet=0.7, queen_diet=7.0, cone_range=4). The profile isn't the issue. The reason you get 11 pawns for 19 hexes is the **per-hex throughput constraint**, not coverage.

Look at the export per pawn by tier:

| Tier | Export (1.3×tier - 0.7) |
|------|------------------------|
| T1   | 0.6                    |
| T2   | 1.9                    |
| T5   | 5.8                    |
| T7   | 8.4                    |

The MIP requires `Σ exports ≥ 7.0` at **every** route hex. In the low-tier middle section (T1/T2 candidates), you need 4+ pawns stacking coverage on each hex just to hit 7.0. That's what's blowing up the count.

The vision-4 range actually *helps* (without it you'd need even more), but it can't overcome the throughput bottleneck in barren sections.

The fix is that the queen **has a food reserve** (capacity=24). She doesn't need 7.0/turn at every hex — she can burn reserves through lean stretches and refill at rich ones. The current MIP is over-constrained. You could relax it to something like:

- **Sliding-window constraint**: over any window of *k* consecutive hexes, total available export ≥ *k* × queen_diet − queen_capacity
- Or simpler: just require that the **cumulative** export along the route never lets the queen's food hit zero, starting from full capacity

That would let T7 pawns at the ends "bank" food, and the queen coasts through the lean middle on reserves — which is exactly what the sim already showed (she arrived fine with food to spare).

the PCR **is** already using `FOOD_PROFILE_V2` values (harvest=1.3, diet=0.7, queen_diet=7.0, cone_range=4). The profile isn't the issue. The reason you get 11 pawns for 19 hexes is the **per-hex throughput constraint**, not coverage.

Look at the export per pawn by tier:

| Tier | Export (1.3×tier - 0.7) |
|------|------------------------|
| T1   | 0.6                    |
| T2   | 1.9                    |
| T5   | 5.8                    |
| T7   | 8.4                    |

The MIP requires `Σ exports ≥ 7.0` at **every** route hex. In the low-tier middle section (T1/T2 candidates), you need 4+ pawns stacking coverage on each hex just to hit 7.0. That's what's blowing up the count.

The vision-4 range actually *helps* (without it you'd need even more), but it can't overcome the throughput bottleneck in barren sections.

The fix is that the queen **has a food reserve** (capacity=24). She doesn't need 7.0/turn at every hex — she can burn reserves through lean stretches and refill at rich ones. The current MIP is over-constrained. You could relax it to something like:

- **Sliding-window constraint**: over any window of *k* consecutive hexes, total available export ≥ *k* × queen_diet − queen_capacity
- Or simpler: just require that the **cumulative** export along the route never lets the queen's food hit zero, starting from full capacity

That would let T7 pawns at the ends "bank" food, and the queen coasts through the lean middle on reserves — which is exactly what the sim already showed (she arrived fine with food to spare).

So I was thinking of a demo with a queen and four pawns. basically she would have two pawns always feed her and  the other two pawns would move forward to set up for the new route. The pawns would know their locations. it is a question of timing things so that the queen has coverage along the way.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a much better design — it turns the PCR from a static placement problem into a **leapfrog relay**. Way more interesting as a game mechanic too.

Let's think through the timing. The key numbers:

- Queen diet: 7.0/turn, capacity: 24 → **~3.4 turns of reserves**
- Queen speed: ~2 hexes/turn (from the sim)
- Pawn speed: ~1 hex/turn (they're slower, right?)
- Pawn sight/feed range: 4 hexes

So the pattern would be something like:

**Station A** — Pawns 1 & 2 on fertile hexes, harvesting + giving to queen as she passes through their coverage cone (4 hex range means they cover ~8 hexes of route from one spot).

**Meanwhile** — Pawns 3 & 4 are marching ahead to **Station B**, the next fertile cluster.

**Gap** — Queen burns reserves crossing any lean stretch between coverage zones. With 3.4 turns of reserves and 2 hex/turn speed, she can bridge ~6-7 barren hexes.

**Swap** — Once queen reaches Station B and Pawns 3 & 4 start feeding, Pawns 1 & 2 leapfrog ahead to Station C.

The tricky constraint is that pawns move slower than the queen. If the queen does 2 hex/turn and pawns do 1, the leapfrog pair needs a **head start** — they need to leave Station A *before* the queen finishes passing through.

Want to start by nailing down the pawn movement speed and working out the exact timing math? Or would you rather jump straight into coding a choreographed 5-piece demo and tune it empirically in the sim?

We can do a non optimized version first of where the queen stays put until the path ahead is clear then goes to the edge of the new coverage and waits there as the pawns travel. There is definitely some extra turns burnt, but the queen will be safe and the algorithm will work.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love it — the "wait until safe" approach is the right first cut. Simple invariant: **the queen never moves unless she can reach a fed position**.

Here's the algorithm sketch:

1. **Find stations** — Walk the route, find pairs of fertile hexes adjacent to route segments, spaced so their coverage cones overlap or the gap ≤ queen's reserve range
2. **Leapfrog loop**:
   - Queen sits at Station N, fed by Pair A
   - Pair B marches to Station N+1, sets up (harvest to fill)
   - Once Pair B is ready → queen moves to edge of Pair B's coverage
   - Pair A now marches to Station N+2
   - Repeat, swapping roles each leg

Let's start with the station-finder — given a route, find the best pawn positions along it:

```python
def find_stations(route, grid, food_tiers, feed_range=4, min_tier=2):
    """Find pawn station pairs along a route.
    
    A station is a pair of fertile hexes adjacent to the route,
    not ON the route, that together cover a stretch of it.
    Returns list of {'hexes': [h1,h2], 'covers': set(route_indices)}.
    """
    route_set = set(route)
    covered_up_to = 0
    stations = []
    
    while covered_up_to < len(route):
        # Find best pair of fertile hexes covering route from covered_up_to onward
        best_pair, best_cover = None, set()
        
        # Candidates: fertile hexes adjacent to route[covered_up_to:]
        cands = []
        seen = set()
        for wi in range(max(0, covered_up_to - feed_range), len(route)):
            for nb in grid.neighborsOf(route[wi]):
                if nb in seen or nb in route_set: continue
                seen.add(nb)
                tier = int(food_tiers[nb]) if 0 <= nb < len(food_tiers) else 0
                if tier < min_tier: continue
                
                hp_nb = grid.index_to_hexposition(nb)
                cover = set()
                for wj, rh in enumerate(route):
                    if hp_nb.distance(grid.index_to_hexposition(rh)) <= feed_range:
                        cover.add(wj)
                export = 1.3 * tier - 0.7
                cands.append({'hex': nb, 'tier': tier, 'cover': cover, 'export': export})
        
        # Greedy: pick the candidate that extends coverage furthest,
        # then pick a partner that maximizes combined export
        cands.sort(key=lambda c: -max(c['cover'] & set(range(covered_up_to, len(route))), default=-1))
        
        if not cands:
            # No more fertile hexes — mark remaining as a gap
            stations.append({'hexes': [], 'covers': set(), 
                             'gap_start': covered_up_to, 'gap_end': len(route)-1})
            break
        
        best_a = cands[0]
        # Find best partner (different hex, overlapping or adjacent coverage)
        best_b = None
        best_combined = 0
        for c in cands[1:]:
            if c['hex'] == best_a['hex']: continue
            combined_cover = best_a['cover'] | c['cover']
            reach = max(combined_cover & set(range(covered_up_to, len(route))), default=-1)
            if reach > best_combined:
                best_combined = reach
                best_b = c
        
        pair_hexes = [best_a['hex']]
        pair_cover = best_a['cover']
        if best_b:
            pair_hexes.append(best_b['hex'])
            pair_cover |= best_b['cover']
        
        stations.append({'hexes': pair_hexes, 'covers': pair_cover})
        covered_up_to = max(pair_cover) + 1 if pair_cover else len(route)
    
    return stations
```

Try running that on the existing route to see what stations it picks? Then we'll build the choreography on top.

Shouldn't find_stations call pawn_covered_route? By design those are good stations.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right — `pawn_covered_route` already does the hard work of finding fertile hexes with good coverage cones along the route. No point re-inventing that.

The flow becomes:

1. **`pawn_covered_route`** → gives us all the good pawn positions + their coverage ranges
2. **Cluster those into stations** — group placements into pairs, ordered along the route
3. **Leapfrog** — pairs hop between stations

```python
def find_stations(pcr_result, pawns_per_station=2):
    """Group PCR placements into ordered stations of pawn-pairs.
    
    Each station is 2 pawn slots that a leapfrog pair occupies.
    Stations are ordered by their coverage midpoint along the route.
    """
    placements = sorted(pcr_result['placements'], 
                        key=lambda p: min(p['coverage']))
    
    stations = []
    i = 0
    while i < len(placements):
        group = placements[i:i + pawns_per_station]
        covers = set()
        for p in group:
            covers |= p['coverage']
        stations.append({
            'hexes': [p['hex'] for p in group],
            'tiers': [p['tier'] for p in group],
            'covers': covers,
            'min_w': min(covers),
            'max_w': max(covers),
        })
        i += pawns_per_station
    
    return stations
```

Then the leapfrog choreographer just needs to walk the queen station-to-station, sending the idle pair ahead each time. Want to try this grouping on the existing PCR result to see if the station spacing looks reasonable for the queen's 3.4-turn reserve?

In [ ]:
def find_stations(pcr_result, pawns_per_station=2):
    """Group PCR placements into ordered stations of pawn-pairs.
    
    Each station is 2 pawn slots that a leapfrog pair occupies.
    Stations are ordered by their coverage midpoint along the route.
    """
    placements = sorted(pcr_result['placements'], 
                        key=lambda p: min(p['coverage']))
    
    stations = []
    i = 0
    while i < len(placements):
        group = placements[i:i + pawns_per_station]
        covers = set()
        for p in group:
            covers |= p['coverage']
        stations.append({
            'hexes': [p['hex'] for p in group],
            'tiers': [p['tier'] for p in group],
            'covers': covers,
            'min_w': min(covers),
            'max_w': max(covers),
        })
        i += pawns_per_station
    
    return stations


So we want to try to keep the 3-4 reserve because she might have a pawn killed and then would be in trouble. We can do a desapartion run later. right now she should be plump. I think we might need a class that coordinates this journey and interacts with the food simulator

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — keep her plump, save the desperate sprint for a "last resort" mechanic later. Here's the coordinator class:

```python
from enum import Enum
from collections import deque

class JourneyPhase(Enum):
    STAGING  = 'staging'   # idle pair marching to next station
    READY    = 'ready'     # next station manned, queen can go
    MOVING   = 'moving'    # queen in transit
    ARRIVED  = 'arrived'   # queen at final destination
    
class LeapfrogJourney:
    """Coordinates queen + 4 pawns in a leapfrog relay.
    
    Invariant: queen never moves unless next station is manned
    and she has >= reserve_turns of food.
    
    Used as a FoodSimulator callback via __call__.
    """
    
    def __init__(self, queen, pair_a, pair_b, route, stations,
                 grid, elevations, reserve_turns=3):
        self.queen = queen
        self.pairs = [list(pair_a), list(pair_b)]
        self.route = route
        self.stations = stations
        self.grid = grid
        self.elevations = elevations
        self.reserve_turns = reserve_turns
        
        # State
        self.active = 0          # index into self.pairs — who's feeding
        self.idle = 1             # who's leapfrogging
        self.station_idx = 0     # queen's current station
        self.phase = JourneyPhase.STAGING
        self.queen_w = 0         # queen's position on route (index)
        self.log = []
        
    @property
    def active_pair(self): return self.pairs[self.active]
    @property
    def idle_pair(self): return self.pairs[self.idle]
    @property
    def next_station_idx(self): return self.station_idx + 1
    
    def _bfs_path(self, start, end):
        """Short BFS between two hexes."""
        if start == end: return [start]
        parent = {start: None}
        q = deque([start])
        while q:
            cur = q.popleft()
            if cur == end: break
            for nb in self.grid.neighborsOf(cur):
                if nb in parent or nb < 0: continue
                if nb >= len(self.elevations) or self.elevations[nb] <= 0: continue
                if nb in self.grid.invalidRegion: continue
                parent[nb] = cur
                q.append(nb)
        if end not in parent: return None
        path = []
        cur = end
        while cur is not None:
            path.append(cur); cur = parent[cur]
        path.reverse()
        return path
    
    def _march_pawn_to(self, pawn, target_hex):
        """Give a pawn instructions to walk to target, then harvest+give."""
        if pawn.location == target_hex:
            self._set_harvest_give(pawn)
            return
        path = self._bfs_path(pawn.location, target_hex)
        if not path or len(path) < 2:
            self._set_harvest_give(pawn)
            return
        rules, _ = route_to_instructions(path, self.grid, pawn.facing)
        # Append harvest+give loop once arrived
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        pawn.instructions = InstructionList(rules, cursor=0, patrol=False)
    
    def _set_harvest_give(self, pawn):
        pawn.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)
    
    def _pair_in_position(self, pair, station_idx):
        """Both pawns at their assigned station hexes?"""
        if station_idx >= len(self.stations): return False
        targets = set(self.stations[station_idx]['hexes'])
        return all(p.location in targets for p in pair)
    
    def _send_idle_to_next(self):
        """Dispatch idle pair to next station."""
        nsi = self.next_station_idx
        if nsi >= len(self.stations): return
        station = self.stations[nsi]
        for i, pawn in enumerate(self.idle_pair):
            target = station['hexes'][i % len(station['hexes'])]
            self._march_pawn_to(pawn, target)
        self._log(f"Pair {self.idle} dispatched to station {nsi}")
    
    def _queen_move_to_station(self, target_station_idx):
        """Give queen march orders to the edge of next station's coverage."""
        cov = self.stations[target_station_idx]['covers']
        target_w = min(cov)  # first route hex in next station's range
        if target_w <= self.queen_w: 
            target_w = max(cov)  # already overlapping, go deeper
        sub = self.route[self.queen_w:target_w + 1]
        if len(sub) > 1:
            rules, _ = route_to_instructions(sub, self.grid, self.queen.facing)
            self.queen.instructions = InstructionList(rules, cursor=0, patrol=False)
            self._log(f"Queen moving route[{self.queen_w}→{target_w}]")
    
    def _update_queen_w(self):
        """Sync queen_w with queen's actual location."""
        for i, rh in enumerate(self.route):
            if rh == self.queen.location:
                self.queen_w = i
                return
    
    def _queen_is_plump(self):
        """Queen has enough reserve to justify moving."""
        return self.queen.food >= self.queen.diet * self.reserve_turns
    
    def _log(self, msg):
        self.log.append(msg)
    
    def __call__(self, report: TurnReport):
        """FoodSimulator callback — runs each turn."""
        # Always share food between adjacent pieces
        food_redistribute_adjacent(report)
        self._update_queen_w()
        
        if self.phase == JourneyPhase.ARRIVED:
            return
        
        if self.phase == JourneyPhase.STAGING:
            if self._pair_in_position(self.idle_pair, self.next_station_idx):
                # Set them harvesting
                for p in self.idle_pair:
                    self._set_harvest_give(p)
                self.phase = JourneyPhase.READY
                self._log(f"Station {self.next_station_idx} ready")
        
        if self.phase == JourneyPhase.READY:
            if self._queen_is_plump():
                self._queen_move_to_station(self.next_station_idx)
                self.phase = JourneyPhase.MOVING
        
        if self.phase == JourneyPhase.MOVING:
            next_cov = self.stations[self.next_station_idx]['covers']
            if self.queen_w in next_cov:
                # Swap pairs
                self.station_idx = self.next_station_idx
                self.active, self.idle = self.idle, self.active
                self._log(f"Queen at station {self.station_idx}, swapped pairs")
                
                if self.next_station_idx >= len(self.stations):
                    self.phase = JourneyPhase.ARRIVED
                    self._log("Journey complete!")
                else:
                    self.phase = JourneyPhase.STAGING
                    self._send_idle_to_next()
    
    def setup(self):
        """Initial setup: place pair A at station 0, send pair B to station 1."""
        s0 = self.stations[0]
        for i, pawn in enumerate(self.active_pair):
            target = s0['hexes'][i % len(s0['hexes'])]
            pawn.location = target
            self._set_harvest_give(pawn)
        
        self.queen.location = self.route[0]
        self._send_idle_to_next()
        self.phase = JourneyPhase.STAGING
        self._log("Journey initialized")
```

The idea: `setup()` places pair A at station 0, dispatches pair B, then you feed this as the callback to `FoodSimulator`. The queen just sits and eats until the next station is manned and she's plump, then goes.

Want to wire it up to the existing route + stations and run it?

In [ ]:
from enum import Enum
from collections import deque

class JourneyPhase(Enum):
    STAGING  = 'staging'   # idle pair marching to next station
    READY    = 'ready'     # next station manned, queen can go
    MOVING   = 'moving'    # queen in transit
    ARRIVED  = 'arrived'   # queen at final destination
    
class LeapfrogJourney:
    """Coordinates queen + 4 pawns in a leapfrog relay.
    
    Invariant: queen never moves unless next station is manned
    and she has >= reserve_turns of food.
    
    Used as a FoodSimulator callback via __call__.
    """
    
    def __init__(self, queen, pair_a, pair_b, route, stations,
                 grid, elevations, reserve_turns=3):
        self.queen = queen
        self.pairs = [list(pair_a), list(pair_b)]
        self.route = route
        self.stations = stations
        self.grid = grid
        self.elevations = elevations
        self.reserve_turns = reserve_turns
        
        # State
        self.active = 0          # index into self.pairs — who's feeding
        self.idle = 1             # who's leapfrogging
        self.station_idx = 0     # queen's current station
        self.phase = JourneyPhase.STAGING
        self.queen_w = 0         # queen's position on route (index)
        self.log = []
        
    @property
    def active_pair(self): return self.pairs[self.active]
    @property
    def idle_pair(self): return self.pairs[self.idle]
    @property
    def next_station_idx(self): return self.station_idx + 1
    
    def _bfs_path(self, start, end):
        """Short BFS between two hexes."""
        if start == end: return [start]
        parent = {start: None}
        q = deque([start])
        while q:
            cur = q.popleft()
            if cur == end: break
            for nb in self.grid.neighborsOf(cur):
                if nb in parent or nb < 0: continue
                if nb >= len(self.elevations) or self.elevations[nb] <= 0: continue
                if nb in self.grid.invalidRegion: continue
                parent[nb] = cur
                q.append(nb)
        if end not in parent: return None
        path = []
        cur = end
        while cur is not None:
            path.append(cur); cur = parent[cur]
        path.reverse()
        return path
    
    def _march_pawn_to(self, pawn, target_hex):
        """Give a pawn instructions to walk to target, then harvest+give."""
        if pawn.location == target_hex:
            self._set_harvest_give(pawn)
            return
        path = self._bfs_path(pawn.location, target_hex)
        if not path or len(path) < 2:
            self._set_harvest_give(pawn)
            return
        rules, _ = route_to_instructions(path, self.grid, pawn.facing)
        # Append harvest+give loop once arrived
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        pawn.instructions = InstructionList(rules, cursor=0, patrol=False)
    
    def _set_harvest_give(self, pawn):
        pawn.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)
    
    def _pair_in_position(self, pair, station_idx):
        """Both pawns at their assigned station hexes?"""
        if station_idx >= len(self.stations): return False
        targets = set(self.stations[station_idx]['hexes'])
        return all(p.location in targets for p in pair)
    
    def _send_idle_to_next(self):
        """Dispatch idle pair to next station."""
        nsi = self.next_station_idx
        if nsi >= len(self.stations): return
        station = self.stations[nsi]
        for i, pawn in enumerate(self.idle_pair):
            target = station['hexes'][i % len(station['hexes'])]
            self._march_pawn_to(pawn, target)
        self._log(f"Pair {self.idle} dispatched to station {nsi}")
    
    def _queen_move_to_station(self, target_station_idx):
        """Give queen march orders to the edge of next station's coverage."""
        cov = self.stations[target_station_idx]['covers']
        target_w = min(cov)  # first route hex in next station's range
        if target_w <= self.queen_w: 
            target_w = max(cov)  # already overlapping, go deeper
        sub = self.route[self.queen_w:target_w + 1]
        if len(sub) > 1:
            rules, _ = route_to_instructions(sub, self.grid, self.queen.facing)
            self.queen.instructions = InstructionList(rules, cursor=0, patrol=False)
            self._log(f"Queen moving route[{self.queen_w}→{target_w}]")
    
    def _update_queen_w(self):
        """Sync queen_w with queen's actual location."""
        for i, rh in enumerate(self.route):
            if rh == self.queen.location:
                self.queen_w = i
                return
    
    def _queen_is_plump(self):
        """Queen has enough reserve to justify moving."""
        return self.queen.food >= self.queen.diet * self.reserve_turns
    
    def _log(self, msg):
        self.log.append(msg)
    
    def __call__(self, report: TurnReport):
        """FoodSimulator callback — runs each turn."""
        # Always share food between adjacent pieces
        food_redistribute_adjacent(report)
        self._update_queen_w()
        
        if self.phase == JourneyPhase.ARRIVED:
            return
        
        if self.phase == JourneyPhase.STAGING:
            if self._pair_in_position(self.idle_pair, self.next_station_idx):
                # Set them harvesting
                for p in self.idle_pair:
                    self._set_harvest_give(p)
                self.phase = JourneyPhase.READY
                self._log(f"Station {self.next_station_idx} ready")
        
        if self.phase == JourneyPhase.READY:
            if self._queen_is_plump():
                self._queen_move_to_station(self.next_station_idx)
                self.phase = JourneyPhase.MOVING
        
        if self.phase == JourneyPhase.MOVING:
            next_cov = self.stations[self.next_station_idx]['covers']
            if self.queen_w in next_cov:
                # Swap pairs
                self.station_idx = self.next_station_idx
                self.active, self.idle = self.idle, self.active
                self._log(f"Queen at station {self.station_idx}, swapped pairs")
                
                if self.next_station_idx >= len(self.stations):
                    self.phase = JourneyPhase.ARRIVED
                    self._log("Journey complete!")
                else:
                    self.phase = JourneyPhase.STAGING
                    self._send_idle_to_next()
    
    def setup(self):
        """Initial setup: place pair A at station 0, send pair B to station 1."""
        s0 = self.stations[0]
        for i, pawn in enumerate(self.active_pair):
            target = s0['hexes'][i % len(s0['hexes'])]
            pawn.location = target
            self._set_harvest_give(pawn)
        
        self.queen.location = self.route[0]
        self._send_idle_to_next()
        self.phase = JourneyPhase.STAGING
        self._log("Journey initialized")


In [ ]:
@patch
def _bfs_path(self:LeapfrogJourney, start, end, avoid=None):
    """Short BFS between two hexes, avoiding occupied hexes."""
    avoid = avoid or set()
    if start == end: return [start]
    parent = {start: None}
    q = deque([start])
    while q:
        cur = q.popleft()
        if cur == end: break
        for nb in self.grid.neighborsOf(cur):
            if nb in parent or nb < 0: continue
            if nb in avoid and nb != end: continue  # can walk TO target, not through
            if nb >= len(self.elevations) or self.elevations[nb] <= 0: continue
            if nb in self.grid.invalidRegion: continue
            parent[nb] = cur
            q.append(nb)
    if end not in parent: return None
    path = []
    cur = end
    while cur is not None:
        path.append(cur); cur = parent[cur]
    path.reverse()
    return path


In [ ]:
@patch
def _occupied_hexes(self:LeapfrogJourney):
    """All hexes occupied by any piece in the journey."""
    occ = set()
    for pair in self.pairs:
        for p in pair:
            if p.location is not None and p.location >= 0:
                occ.add(p.location)
    if self.queen.location is not None:
        occ.add(self.queen.location)
    return occ

@patch
def _march_pawn_to(self:LeapfrogJourney, pawn, target_hex):
    if pawn.location == target_hex:
        self._set_harvest_give(pawn)
        return
    avoid = self._occupied_hexes() - {pawn.location, target_hex}
    path = self._bfs_path(pawn.location, target_hex, avoid=avoid)
    if not path or len(path) < 2:
        self._set_harvest_give(pawn)
        return
    rules, _ = route_to_instructions(path, self.grid, pawn.facing)
    rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
    pawn.instructions = InstructionList(rules, cursor=0, patrol=False)

In [ ]:
@patch
def setup(self: LeapfrogJourney):
    """Place pair A at station 0, queen adjacent to them, send pair B ahead."""
    s0 = self.stations[0]
    for i, pawn in enumerate(self.active_pair):
        target = s0['hexes'][i % len(s0['hexes'])]
        pawn.location = target
        self._set_harvest_give(pawn)
    
    # Place queen adjacent to first pawn of active pair, ON the route
    best_start = self.route[0]
    for rh in self.route:
        if self.grid.neighborsOf(rh) and \
           any(rh in self.grid.neighborsOf(p.location) or rh == p.location 
               for p in self.active_pair):
            best_start = rh
            break
    
    self.queen.location = best_start
    self._update_queen_w()
    self._send_idle_to_next()
    self.phase = JourneyPhase.STAGING
    self._log(f"Journey initialized, queen at ⬡{best_start} (route[{self.queen_w}])")


Can you make the queen now travel across the board? This should be the same route as the earlier demo between cities.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# ── Build stations from the existing PCR ──
stations = find_stations(pcr)
print(f"{len(stations)} stations:")
for i, s in enumerate(stations):
    print(f"  Station {i}: hexes={s['hexes']} tiers={s['tiers']} "
          f"covers route[{s['min_w']}..{s['max_w']}]")

# ── Create fresh squad: queen + 4 pawns ──
lf_squad = Squad.squads(1, myStuff.country.flag)[0]

queen = myStuff.home.recruit(PieceType.QUEEN, flag=lf_squad.flag)
lf_squad.pieces.append(queen)

pawns = []
for i in range(4):
    p = myStuff.home.recruit(PieceType.PAWN, flag=lf_squad.flag)
    pawns.append(p)
    lf_squad.pieces.append(p)

apply_food_profile(lf_squad.pieces, FOOD_PROFILE_V2)

pair_a = pawns[0:2]
pair_b = pawns[2:4]

# ── Create the journey ──
journey = LeapfrogJourney(
    queen=queen,
    pair_a=pair_a,
    pair_b=pair_b,
    route=pcr['route'],
    stations=stations,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    reserve_turns=3,
)
journey.setup()

print(f"\nQueen at ⬡{queen.location}, food={queen.food:.1f}")
for i, p in enumerate(pawns):
    print(f"  Pawn {i} ({p.name}): ⬡{p.location} → {p.instructions.rules[:5]}...")

# ── Run simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=lf_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[lf_squad],
    callbacks={lf_squad.id: journey},
)

sim.run(num_turns=80)
sim.summary()

# ── Journey log ──
print(f"\nPhase: {journey.phase.value}")
print(f"Queen at ⬡{queen.location} (route[{journey.queen_w}]), food={queen.food:.1f}")
print(f"Target: ⬡{pcr['route'][-1]}")
print(f"\nJourney log:")
for msg in journey.log:
    print(f"  {msg}")
```

This wires everything together — the stations come from the PCR, the journey coordinator handles all the choreography as a simulator callback. 80 turns should be plenty for a 19-hex route at cautious pace. We'll see the log showing each phase transition: staging → ready → moving → swap → repeat.

In [ ]:
# ── Build stations from the existing PCR ──
stations = find_stations(pcr)
print(f"{len(stations)} stations:")
for i, s in enumerate(stations):
    print(f"  Station {i}: hexes={s['hexes']} tiers={s['tiers']} "
          f"covers route[{s['min_w']}..{s['max_w']}]")

# ── Create fresh squad: queen + 4 pawns ──
lf_squad = Squad.squads(1, myStuff.country.flag)[0]

queen = myStuff.home.recruit(PieceType.QUEEN, flag=lf_squad.flag)
lf_squad.pieces.append(queen)

pawns = []
for i in range(4):
    p = myStuff.home.recruit(PieceType.PAWN, flag=lf_squad.flag)
    pawns.append(p)
    lf_squad.pieces.append(p)

apply_food_profile(lf_squad.pieces, FOOD_PROFILE_V2)

pair_a = pawns[0:2]
pair_b = pawns[2:4]

# ── Create the journey ──
journey = LeapfrogJourney(
    queen=queen,
    pair_a=pair_a,
    pair_b=pair_b,
    route=pcr['route'],
    stations=stations,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    reserve_turns=3,
)
journey.setup()

print(f"\nQueen at ⬡{queen.location}, food={queen.food:.1f}")
for i, p in enumerate(pawns):
    print(f"  Pawn {i} ({p.name}): ⬡{p.location} → {p.instructions.rules[:5]}...")

# ── Run simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=lf_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[lf_squad],
    callbacks={lf_squad.id: journey},
)

sim.run(num_turns=80)
sim.summary()

# ── Journey log ──
print(f"\nPhase: {journey.phase.value}")
print(f"Queen at ⬡{queen.location} (route[{journey.queen_w}]), food={queen.food:.1f}")
print(f"Target: ⬡{pcr['route'][-1]}")
print(f"\nJourney log:")
for msg in journey.log:
    print(f"  {msg}")


We don't get to station 2? is a pawn trying to move through the queen?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two issues jumping out:

**1. Rose moved 0 hexes** — that's the blocker. The journey is stuck in STAGING because `_pair_in_position` needs *both* pawns at station 1, and Rose never arrived at ⬡276. Her instructions `[6, 5, 7, 5, 5]...` look like mostly rotations. The BFS path → `route_to_instructions` conversion may be failing to match BFS neighbors to `HexPosition.directions()`, causing skipped steps. Worth a quick sanity check:

```python
# Debug: check Rose's BFS path and instruction conversion
path_rose = journey._bfs_path(158, 276)
print(f"BFS 158→276: {path_rose} ({len(path_rose)} hexes)")
for i in range(len(path_rose)-1):
    nbs = myStuff.grid.neighborsOf(path_rose[i])
    print(f"  {path_rose[i]}→{path_rose[i+1]}: in neighbors? {path_rose[i+1] in nbs}")
```

**2. Too many stations** — the PCR gave us 11 placements grouped into 6 stations, but we only have 4 pawns (2 pairs). With leapfrog we only need ~3 well-spaced stations. Stations 0 and 1 overlap massively (both cover route[0..6]), so pair B is walking to a nearly-redundant position.

We should pick a **subset** — something like every-other or filter by minimum gap:

```python
def select_leapfrog_stations(stations, queen_reserve_hexes=6, min_count=3):
    """Pick well-spaced stations for leapfrog. 
    Drop stations whose coverage mostly overlaps the previous one."""
    selected = [stations[0]]
    for s in stations[1:]:
        prev = selected[-1]
        gap = s['min_w'] - prev['max_w']
        if gap >= -2:  # minimal overlap or actual gap
            selected.append(s)
    # Always include the last station
    if selected[-1] is not stations[-1]:
        selected.append(stations[-1])
    return selected
```

Fix both and the queen should start moving. Want to debug Rose's path first, or skip ahead to the station filtering?

In [ ]:
# Debug: check Rose's BFS path and instruction conversion
path_rose = journey._bfs_path(158, 276)
print(f"BFS 158→276: {path_rose} ({len(path_rose)} hexes)")
for i in range(len(path_rose)-1):
    nbs = myStuff.grid.neighborsOf(path_rose[i])
    print(f"  {path_rose[i]}→{path_rose[i+1]}: in neighbors? {path_rose[i+1] in nbs}")


We can cut down stations later but every station should get you a bit further, which isn't what is happening.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The phase never leaves STAGING because Rose never arrives at ⬡276 — 0 hexes in 80 turns. Two likely culprits:

**1. Three pieces stacked on ⬡158** — `setup()` never places the idle pair. They default to the settlement hex (158), same as the queen. If the sim blocks FORWARD when the destination is occupied, Rose is stuck behind Marie who's stuck behind the queen.

**2. `route_to_instructions` direction mismatch** — let's verify:

```python
# Check what instructions actually get generated for Rose's path
path_rose = [158, 179, 198, 218, 237, 257, 276]
rules, final_f = route_to_instructions(path_rose, myStuff.grid, 0)
print(f"Rules: {rules}")
print(f"FORWARD count: {rules.count(Instruction.FORWARD.value)}")

# Also check Rose's full instruction list and snapshots
print(f"\nRose full instructions: {pawns[3].instructions.rules}")
print(f"Rose cursor: {pawns[3].instructions.cursor}")

# Check if Rose ever moved in snapshots
rose_id = pawns[3].id
rose_moves = sim.df[(sim.df['piece_id'] == rose_id) & (sim.df['event'] == 'moved')]
print(f"\nRose move events: {len(rose_moves)}")
if len(rose_moves) > 0:
    print(rose_moves[['turn', 'src_hex', 'dst_hex']].head(10))

# And where was Rose each turn?
rose_locs = []
for snap in sim.snapshots[:10]:
    rs = snap['piece_states'].get(rose_id)
    if rs: rose_locs.append((snap['turn'], rs['location']))
print(f"\nRose locations: {rose_locs}")
```

My bet is problem #1 — the idle pair needs placement in `setup()`. They should start somewhere near station 0, not on top of the queen.

In [ ]:
# Check what instructions actually get generated for Rose's path
path_rose = [158, 179, 198, 218, 237, 257, 276]
rules, final_f = route_to_instructions(path_rose, myStuff.grid, 0)
print(f"Rules: {rules}")
print(f"FORWARD count: {rules.count(Instruction.FORWARD.value)}")

# Also check Rose's full instruction list and snapshots
print(f"\nRose full instructions: {pawns[3].instructions.rules}")
print(f"Rose cursor: {pawns[3].instructions.cursor}")

# Check if Rose ever moved in snapshots
rose_id = pawns[3].id
rose_moves = sim.df[(sim.df['piece_id'] == rose_id) & (sim.df['event'] == 'moved')]
print(f"\nRose move events: {len(rose_moves)}")
if len(rose_moves) > 0:
    print(rose_moves[['turn', 'src_hex', 'dst_hex']].head(10))

# And where was Rose each turn?
rose_locs = []
for snap in sim.snapshots[:10]:
    rs = snap['piece_states'].get(rose_id)
    if rs: rose_locs.append((snap['turn'], rs['location']))
print(f"\nRose locations: {rose_locs}")


Do we have a path planning problem?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! Look at what happens:

- Rose's path is `158 → 179 → 198 → 218 → 237 → 257 → 276`
- She rotated once (cursor went from 0 to 1) then got stuck on the FORWARD (instruction value `5`)
- Kenneth is **parked at ⬡179** harvesting — that's Rose's very first step

Rose can't walk through Kenneth. The BFS doesn't know about occupied hexes.

The fix for `_bfs_path` — pass in occupied hexes to avoid:

```python
def _bfs_path(self, start, end, avoid=None):
    """Short BFS between two hexes, avoiding occupied hexes."""
    avoid = avoid or set()
    if start == end: return [start]
    parent = {start: None}
    q = deque([start])
    while q:
        cur = q.popleft()
        if cur == end: break
        for nb in self.grid.neighborsOf(cur):
            if nb in parent or nb < 0: continue
            if nb in avoid and nb != end: continue  # can walk TO target, not through
            if nb >= len(self.elevations) or self.elevations[nb] <= 0: continue
            if nb in self.grid.invalidRegion: continue
            parent[nb] = cur
            q.append(nb)
    if end not in parent: return None
    path = []
    cur = end
    while cur is not None:
        path.append(cur); cur = parent[cur]
    path.reverse()
    return path
```

And `_march_pawn_to` / `_send_idle_to_next` should gather all occupied hexes:

```python
def _occupied_hexes(self):
    """All hexes occupied by any piece in the journey."""
    occ = set()
    for pair in self.pairs:
        for p in pair:
            if p.location is not None and p.location >= 0:
                occ.add(p.location)
    if self.queen.location is not None:
        occ.add(self.queen.location)
    return occ

def _march_pawn_to(self, pawn, target_hex):
    if pawn.location == target_hex:
        self._set_harvest_give(pawn)
        return
    avoid = self._occupied_hexes() - {pawn.location, target_hex}
    path = self._bfs_path(pawn.location, target_hex, avoid=avoid)
    if not path or len(path) < 2:
        self._set_harvest_give(pawn)
        return
    rules, _ = route_to_instructions(path, self.grid, pawn.facing)
    rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
    pawn.instructions = InstructionList(rules, cursor=0, patrol=False)
```

Did I patch things in correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, the patches look correct. The `@patch` from fastcore will override the original methods on the class. Three things are properly wired:

1. **`_bfs_path`** — now takes `avoid=None` and skips those hexes (except the destination itself) ✅
2. **`_occupied_hexes`** — new method, gathers all piece locations ✅  
3. **`_march_pawn_to`** — calls `_occupied_hexes()`, excludes self + target, passes to `_bfs_path` ✅

One thing to note: the original class body (cell `_3cb09b17`) still has the old `_bfs_path` and `_march_pawn_to` — that's fine as long as the patches run *after* the class definition (which they do, being below it). But if you eventually move this to a module, you'll want to fold these fixes into the class itself.

Re-run the demo cell to test:

```python
# Re-run with patches applied
rebootWorld()

pcr = pawn_covered_route(
    cities[0].location, cities[1].location,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

stations = find_stations(pcr)
```

Then recreate the squad + journey and run. The idle pair should now route *around* the active pair at station 0.

# Re-run with patches applied
rebootWorld()

pcr = pawn_covered_route(
    cities[0].location, cities[1].location,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

stations = find_stations(pcr)


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That should work as a code cell — but you'll need to recompute `fy` after `rebootWorld()` since it creates a new terrain:

```python
# Re-run with patches applied
rebootWorld()

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

pcr = pawn_covered_route(
    cities[0].location, cities[1].location,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

stations = find_stations(pcr)
print(f"{len(stations)} stations:")
for i, s in enumerate(stations):
    print(f"  Station {i}: hexes={s['hexes']} tiers={s['tiers']} "
          f"covers route[{s['min_w']}..{s['max_w']}]")
```

Then in the next cell, recreate the squad + journey + sim:

```python
# Fresh squad
lf_squad = Squad.squads(1, myStuff.country.flag)[0]
queen = myStuff.home.recruit(PieceType.QUEEN, flag=lf_squad.flag)
lf_squad.pieces.append(queen)

pawns = []
for i in range(4):
    p = myStuff.home.recruit(PieceType.PAWN, flag=lf_squad.flag)
    pawns.append(p)
    lf_squad.pieces.append(p)

apply_food_profile(lf_squad.pieces, FOOD_PROFILE_V2)

journey = LeapfrogJourney(
    queen=queen, pair_a=pawns[0:2], pair_b=pawns[2:4],
    route=pcr['route'], stations=stations,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    reserve_turns=3,
)
journey.setup()

sim = FoodSimulator(
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, pieces=lf_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[lf_squad],
    callbacks={lf_squad.id: journey},
)

sim.run(num_turns=80)
sim.summary()

print(f"\nPhase: {journey.phase.value}")
print(f"Queen at ⬡{queen.location} (route[{journey.queen_w}]), food={queen.food:.1f}")
print(f"\nJourney log:")
for msg in journey.log:
    print(f"  {msg}")
```

The key difference now: `_march_pawn_to` will route the idle pair *around* occupied hexes instead of getting stuck behind them.

In [ ]:
# Re-run with patches applied
rebootWorld()

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

pcr = pawn_covered_route(
    cities[0].location, cities[1].location,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

stations = find_stations(pcr)
print(f"{len(stations)} stations:")
for i, s in enumerate(stations):
    print(f"  Station {i}: hexes={s['hexes']} tiers={s['tiers']} "
          f"covers route[{s['min_w']}..{s['max_w']}]")


In [ ]:
# Fresh squad
lf_squad = Squad.squads(1, myStuff.country.flag)[0]
queen = myStuff.home.recruit(PieceType.QUEEN, flag=lf_squad.flag)
lf_squad.pieces.append(queen)

pawns = []
for i in range(4):
    p = myStuff.home.recruit(PieceType.PAWN, flag=lf_squad.flag)
    pawns.append(p)
    lf_squad.pieces.append(p)

apply_food_profile(lf_squad.pieces, FOOD_PROFILE_V2)

journey = LeapfrogJourney(
    queen=queen, pair_a=pawns[0:2], pair_b=pawns[2:4],
    route=pcr['route'], stations=stations,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    reserve_turns=3,
)
journey.setup()

sim = FoodSimulator(
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, pieces=lf_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[lf_squad],
    callbacks={lf_squad.id: journey},
)

sim.run(num_turns=80)
sim.summary()

print(f"\nPhase: {journey.phase.value}")
print(f"Queen at ⬡{queen.location} (route[{journey.queen_w}]), food={queen.food:.1f}")
print(f"\nJourney log:")
for msg in journey.log:
    print(f"  {msg}")


In [ ]:
# Check what instructions actually get generated for Rose's path
path_rose = [158, 179, 198, 218, 237, 257, 276]
rules, final_f = route_to_instructions(path_rose, myStuff.grid, 0)
print(f"Rules: {rules}")
print(f"FORWARD count: {rules.count(Instruction.FORWARD.value)}")

# Also check Rose's full instruction list and snapshots
print(f"\nRose full instructions: {pawns[3].instructions.rules}")
print(f"Rose cursor: {pawns[3].instructions.cursor}")

# Check if Rose ever moved in snapshots
rose_id = pawns[3].id
rose_moves = sim.df[(sim.df['piece_id'] == rose_id) & (sim.df['event'] == 'moved')]
print(f"\nRose move events: {len(rose_moves)}")
if len(rose_moves) > 0:
    print(rose_moves[['turn', 'src_hex', 'dst_hex']].head(10))

# And where was Rose each turn?
rose_locs = []
for snap in sim.snapshots[:10]:
    rs = snap['piece_states'].get(rose_id)
    if rs: rose_locs.append((snap['turn'], rs['location']))
print(f"\nRose locations: {rose_locs}")

?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

OK so Rose *did* arrive at ⬡276 by turn 5 — the avoid-fix worked. And station 1 is marked `ready`. The problem is simpler now:

**The queen is stuck because she's not plump enough to move.**

- Plump threshold: `7.0 diet × 3 reserve = 21.0`
- Queen's food: `14.0`
- Queen's capacity: `24.0`

She's at ⬡158. The active pair is at ⬡179 and ⬡218. Those are **not adjacent** to her — so `food_redistribute_adjacent` never transfers food. She's slowly starving on self-harvest alone (0.2 × whatever tier 158 is).

The queen needs to start **next to** one of the active pair's pawns, or the active pair needs to be adjacent to the route start. In `setup()`, the queen is placed at `route[0]` (⬡158) but station 0's hexes are ⬡179 and ⬡218 — neither is a neighbor.

Quick fix for `setup()`:

```python
@patch
def setup(self: LeapfrogJourney):
    """Place pair A at station 0, queen adjacent to them, send pair B ahead."""
    s0 = self.stations[0]
    for i, pawn in enumerate(self.active_pair):
        target = s0['hexes'][i % len(s0['hexes'])]
        pawn.location = target
        self._set_harvest_give(pawn)
    
    # Place queen adjacent to first pawn of active pair, ON the route
    best_start = self.route[0]
    for rh in self.route:
        if self.grid.neighborsOf(rh) and \
           any(rh in self.grid.neighborsOf(p.location) or rh == p.location 
               for p in self.active_pair):
            best_start = rh
            break
    
    self.queen.location = best_start
    self._update_queen_w()
    self._send_idle_to_next()
    self.phase = JourneyPhase.STAGING
    self._log(f"Journey initialized, queen at ⬡{best_start} (route[{self.queen_w}])")
```